# Признаки белковых последовательностей

Архив `t6se-training-data.zip` должен лежать в разделе «Файлы». Запустите все ячейки.
Посчитаем AAC, DPC, QSO, CTDC и CTDT и сохраним результаты в CSV.


In [ ]:
import io
import hashlib
import shutil
import zipfile
import urllib.request
from pathlib import Path

import pandas as pd
from google.colab import files

url = "https://raw.githubusercontent.com/fastyboom-netizen/bastion6-protein-features/main/protein_features.py"
urllib.request.urlretrieve(url, "protein_features.py")
from protein_features import calculate_features, parse_fasta


Читаем архив из файлов Colab.

In [ ]:
archive_path = Path("/content/t6se-training-data.zip")
archive_data = archive_path.read_bytes()

records = []
with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
    for name in sorted(archive.namelist()):
        path = Path(name)
        if "__MACOSX" in path.parts or path.name.startswith("._"):
            continue
        if path.suffix.lower() not in {".fasta", ".fa", ".faa"}:
            continue
        if "_pos_" in path.stem.lower():
            class_name, label = "positive", 1
        elif "_neg_" in path.stem.lower():
            class_name, label = "negative", 0
        else:
            raise ValueError(f"Неизвестный класс: {name}")

        text = archive.read(name).decode("utf-8-sig")
        for number, (seq_id, description, sequence) in enumerate(parse_fasta(text), 1):
            records.append({
                "record_key": f"{class_name}_{len(records)+1:05d}",
                "sequence_id": seq_id, "description": description,
                "class": class_name, "label": label, "length": len(sequence),
                "source_file": name, "record_in_file": number,
                "sequence_sha256": hashlib.sha256(sequence.encode()).hexdigest(),
                "sequence": sequence,
            })

sequences = pd.DataFrame(records)
if sequences.empty:
    raise ValueError("В архиве не найдены последовательности")
assert sequences["class"].value_counts().to_dict() == {"negative": 1112, "positive": 138}
sequences.groupby("class")["length"].agg(["count", "min", "max"])


Расчёт признаков. Для QSO используем `maxlag=30` и `weight=0.1`.

In [ ]:
features = pd.DataFrame([
    calculate_features(sequence, maxlag=30, weight=0.1)
    for sequence in sequences["sequence"]
])
assert features.shape == (1250, 562)
assert not features.isna().any().any()

metadata = sequences.drop(columns="sequence")
result = pd.concat([metadata, features], axis=1)
print(f"Готово: {len(result)} белков, {features.shape[1]} признака")

# Один положительный и один отрицательный пример каждого блока.
examples = result.groupby("class", sort=False).head(1)
blocks = ["AAC", "DPC", "QSO", "CTDC", "CTDT"]
for block in blocks:
    columns = [c for c in features if c.startswith(block + "_")]
    print(block, len(columns))
    display(examples[["sequence_id", "class"] + columns[:5]])


Сохраняем CSV и скачиваем архив.

In [ ]:
output = Path("results")
output.mkdir(exist_ok=True)
meta_columns = list(metadata.columns)

def save_csv(table, name):
    table.to_csv(output / name, index=False, encoding="utf-8-sig", float_format="%.17g")

save_csv(result, "all_features.csv")
save_csv(sequences, "sequences.csv")
for class_name in ["positive", "negative"]:
    subset = result[result["class"] == class_name]
    save_csv(subset, f"{class_name}_features.csv")
    for block in blocks:
        columns = [c for c in features if c.startswith(block + "_")]
        save_csv(subset[meta_columns + columns], f"{class_name}_{block}.csv")

for block in blocks:
    columns = [c for c in features if c.startswith(block + "_")]
    save_csv(examples[meta_columns + columns], f"examples_{block}.csv")

shutil.make_archive("bastion6_results", "zip", output)
files.download("bastion6_results.zip")
